### HW 1

In [ ]:
import re
import string

def clean(inp: str) -> str:
    inp = inp.translate(str.maketrans(string.punctuation, " "*len(string.punctuation)))
    inp = re.sub(r'\s+', ' ', inp.lower())
    return inp

In [ ]:
test_sample = "Hej, i'd like to inform, that now I've been promoted to Senior Data Scientist. Thank you to everyone, who supports me."
cleaned_data = clean(test_sample)
cleaned_data

'hej i d like to inform that now i ve been promoted to senior data scientist thank you to everyone who supports me '

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)  # Embedding layer
        self.linear = nn.Linear(embedding_dim, vocab_size)  # Output layer

    def forward(self, x):
        x = self.embeddings(x)
        x = x.mean(dim=1)
        x = self.linear(x)
        return x

def manual_train_test_split(X, y, test_size=0.2, random_seed=1):
    np.random.seed(random_seed)
    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split_idx = int(len(X) * (1 - test_size))
    train_indices, test_indices = indices[:split_idx], indices[split_idx:]

    X_train, X_test = [X[i] for i in train_indices], [X[i] for i in test_indices]
    y_train, y_test = [y[i] for i in train_indices], [y[i] for i in test_indices]

    return X_train, X_test, y_train, y_test

def train(data: str):
    """
    Train CBOW model using PyTorch.
    return: model (trained CBOW model), word2id (mapping from words to indices)
    """
    # ==========================
    # Set up
    # ==========================
    vector_size = 300
    window_size = 2
    num_epochs = 10
    batch_size = 64
    learning_rate = 0.01

    # ==========================
    # Prepare vocabularies
    # ==========================
    corpus = data.split(' ')
    tokens = list(set(corpus))  # Unique words
    word2id = {word: id for id, word in enumerate(tokens)}
    id2word = {id: word for word, id in word2id.items()}
    vocab_size = len(tokens)

    # ==========================
    # Prepare training data
    # ==========================
    X = []
    y = []

    for i in range(len(corpus)):
        wp = i
        lp = max(0, wp - window_size)
        rp = wp + window_size + 1

        _y = corpus[wp]
        _X = [*corpus[lp:wp], *corpus[wp+1:rp]]

        # Convert words to indices
        X_indices = [word2id[word] for word in _X if word in word2id]
        if len(X_indices) == 0:
            continue

        X.append(X_indices)
        y.append(word2id[_y])

    # Convert to tensors
    X_train, X_test, y_train, y_test = manual_train_test_split(X, y)

    X_train = [torch.tensor(x, dtype=torch.long) for x in X_train]
    X_test = [torch.tensor(x, dtype=torch.long) for x in X_test]
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test = torch.tensor(y_test, dtype=torch.long)


    # ==========================
    # Model Setup
    # ==========================
    model = CBOWModel(vocab_size, vector_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # ==========================
    # Training Loop
    # ==========================
    for epoch in range(num_epochs):
        total_loss = 0

        for i in range(len(X_train)):
            optimizer.zero_grad()

            input_tensor = X_train[i].unsqueeze(0)  # (1, num_context_words)
            target_tensor = y_train[i].unsqueeze(0)  # (1,)

            output = model(input_tensor)  # Forward pass
            loss = criterion(output, target_tensor)  # Compute loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update weights

            total_loss += loss.item()

        # print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

    # ==========================
    # Evaluate Model (Log Loss)
    # ==========================
    # with torch.no_grad():
    #     y_pred = []
    #     for i in range(len(X_test)):
    #         input_tensor = X_test[i].unsqueeze(0)
    #         output = model(input_tensor)
    #         probs = torch.softmax(output, dim=1)  # Convert logits to probabilities
    #         y_pred.append(probs.squeeze(0).numpy())

    # y_pred = np.array(y_pred)
    # logg_loss = log_loss(y_test.numpy(), y_pred, labels=np.arange(vocab_size))
    # print(f"Final Log Loss: {logg_loss}")

    word_embeddings = model.embeddings.weight.data
    res = {word: word_embeddings[idx-1].numpy() for word, idx in word2id.items()}

    return res

# Example usage
res = train(cleaned_data)

Epoch 1/100, Loss: 86.5680
Epoch 2/100, Loss: 5.5935
Epoch 3/100, Loss: 3.1102
Epoch 4/100, Loss: 1.0072
Epoch 5/100, Loss: 0.5513
Epoch 6/100, Loss: 0.4148
Epoch 7/100, Loss: 0.3395
Epoch 8/100, Loss: 0.2868
Epoch 9/100, Loss: 0.2467
Epoch 10/100, Loss: 0.2149
Epoch 11/100, Loss: 0.1893
Epoch 12/100, Loss: 0.1682
Epoch 13/100, Loss: 0.1506
Epoch 14/100, Loss: 0.1357
Epoch 15/100, Loss: 0.1230
Epoch 16/100, Loss: 0.1121
Epoch 17/100, Loss: 0.1026
Epoch 18/100, Loss: 0.0944
Epoch 19/100, Loss: 0.0871
Epoch 20/100, Loss: 0.0807
Epoch 21/100, Loss: 0.0749
Epoch 22/100, Loss: 0.0698
Epoch 23/100, Loss: 0.0652
Epoch 24/100, Loss: 0.0611
Epoch 25/100, Loss: 0.0574
Epoch 26/100, Loss: 0.0540
Epoch 27/100, Loss: 0.0509
Epoch 28/100, Loss: 0.0480
Epoch 29/100, Loss: 0.0454
Epoch 30/100, Loss: 0.0430
Epoch 31/100, Loss: 0.0408
Epoch 32/100, Loss: 0.0388
Epoch 33/100, Loss: 0.0369
Epoch 34/100, Loss: 0.0352
Epoch 35/100, Loss: 0.0336
Epoch 36/100, Loss: 0.0320
Epoch 37/100, Loss: 0.0306
Epoch 38/

In [ ]:
import torch
import torch.nn.functional as F

def find_similar_words(target_word, word_vectors, top_n=5):
    if target_word not in word_vectors:
        print(f"'{target_word}' not found in vocabulary.")
        return []

    # Convert dictionary values (vectors) into a tensor
    words = list(word_vectors.keys())
    vectors = torch.stack([torch.tensor(word_vectors[word]) for word in words])

    # Get the target word vector and ensure it's a tensor
    target_vector = torch.tensor(word_vectors[target_word]).unsqueeze(0)  # Shape (1, vector_size)

    # Compute cosine similarity between the target word and all other words
    similarities = F.cosine_similarity(target_vector, vectors, dim=1)

    # Get sorted indices, excluding the target word itself
    sorted_indices = similarities.argsort(descending=True).tolist()
    sorted_indices.remove(words.index(target_word))  # Remove self from similarity list

    # Take top N similar words
    similar_words = [(words[idx], similarities[idx].item()) for idx in sorted_indices[:top_n]]

    return similar_words


corpus = cleaned_data.split(' ')
tokens = list(set(corpus))
word2id = {word: id for id, word in enumerate(tokens)}
id2word = {id: word for word, id in word2id.items()}

find_similar_words("hej", res)

[('promoted', 0.052550651133060455),
 ('supports', 0.040633801370859146),
 ('i', 0.03403318673372269),
 ('been', 0.018372917547822),
 ('you', 0.002694176509976387)]

In [ ]:
y_train

tensor([ 7,  0,  6,  1,  5,  3,  9,  8,  4, 10,  6,  2])